In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Tüm paketler yüklendi ✓")
print(f"Pandas sürümü: {pd.__version__}")
print(f"NumPy sürümü: {np.__version__}")

Tüm paketler yüklendi ✓
Pandas sürümü: 3.0.2
NumPy sürümü: 2.4.4


In [2]:
from pathlib import Path

# Proje klasörüne göre otomatik yol bulma
PROJECT_ROOT = Path.cwd().parent  # notebooks klasöründen bir üst klasöre çık
DATA_RAW = PROJECT_ROOT / "data" / "raw"

# Dosyaları tanımla
cognitive_file = DATA_RAW / "CY08MSP_STU_COG.SAV"
student_file = DATA_RAW / "CY08MSP_STU_QQQ.SAV"
timing_file = DATA_RAW / "CY08MSP_STU_TIM.SAV"

# Dosyaların var olduğunu kontrol et
for f in [cognitive_file, student_file, timing_file]:
    if f.exists():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"✓ {f.name} — {size_mb:.1f} MB")
    else:
        print(f"✗ BULUNAMADI: {f.name}")

✓ CY08MSP_STU_COG.SAV — 3560.8 MB
✓ CY08MSP_STU_QQQ.SAV — 1999.5 MB
✓ CY08MSP_STU_TIM.SAV — 633.0 MB


In [3]:
import pyreadstat

# Dosyayı tam okumadan sadece değişken adlarını ve etiketlerini alalım
# Bu işlem saniyeler sürer, tüm dosyayı yüklemez
_, meta_cog = pyreadstat.read_sav(str(cognitive_file), metadataonly=True)

print(f"Cognitive dosyasında toplam değişken sayısı: {len(meta_cog.column_names)}")
print(f"İlk 20 değişken:")
for name in meta_cog.column_names[:20]:
    label = meta_cog.column_names_to_labels.get(name, "")
    print(f"  {name}: {label}")

Cognitive dosyasında toplam değişken sayısı: 5023
İlk 20 değişken:
  CNT: Country code 3-character
  CNTRYID: Country Identifier
  CNTSCHID: Intl. School ID
  CNTSTUID: Intl. Student ID
  CYC: PISA Assessment Cycle (2 digits + 2 character Assessment type - MS/FT)
  NatCen: National Centre 6-digit Code
  STRATUM: Stratum ID 5-character (cnt + original stratum ID)
  SUBNATIO: Adjudicated sub-region code 7-digit code (3-digit country code + region ID + stratum ID)
  REGION: REGION
  OECD: OECD country
  ADMINMODE: Mode of Respondent
  LANGTEST_COG: Language of Assessment
  Option_CT: Creative Thinking Option
  Option_FL: Financial Literacy Option
  Option_UH: Une Heure Option
  BOOKID: Form Identifier
  RDESIGN: Reading Adaptive Design (A or B)
  RCORE_TEST: Reading Adaptive Testlet Assignment: Core
  RCORE_PERF: Reading Adaptive Performance: Core
  RS1_LEV: Reading Adaptive Level Assignment: Stage1


In [4]:
_, meta_stu = pyreadstat.read_sav(str(student_file), metadataonly=True)

print(f"Student dosyasında toplam değişken sayısı: {len(meta_stu.column_names)}")
print(f"\nÜlke kodu (CNT) ve ülke adı (CNTRYID) nerede?")
for name in ['CNT', 'CNTRYID', 'CNTSCHID', 'CNTSTUID']:
    if name in meta_stu.column_names_to_labels:
        print(f"  {name}: {meta_stu.column_names_to_labels[name]}")

Student dosyasında toplam değişken sayısı: 1278

Ülke kodu (CNT) ve ülke adı (CNTRYID) nerede?
  CNT: Country code 3-character
  CNTRYID: Country Identifier
  CNTSCHID: Intl. School ID
  CNTSTUID: Intl. Student ID


In [5]:
# CNT değişkeninin değer etiketlerini al - bu ülkelerin listesi
if 'CNT' in meta_stu.variable_value_labels:
    countries = meta_stu.variable_value_labels['CNT']
    print(f"Toplam {len(countries)} ülke/ekonomi var:\n")
    for code, name in sorted(countries.items()):
        print(f"  {code}: {name}")

Toplam 81 ülke/ekonomi var:

  ALB: Albania
  ARE: United Arab Emirates
  ARG: Argentina
  AUS: Australia
  AUT: Austria
  BEL: Belgium
  BGR: Bulgaria
  BRA: Brazil
  BRN: Brunei Darussalam
  CAN: Canada
  CHE: Switzerland
  CHL: Chile
  COL: Colombia
  CRI: Costa Rica
  CZE: Czech Republic
  DEU: Germany
  DNK: Denmark
  DOM: Dominican Republic
  ESP: Spain
  EST: Estonia
  FIN: Finland
  FRA: France
  GBR: United Kingdom
  GEO: Georgia
  GRC: Greece
  GTM: Guatemala
  HKG: Hong Kong (China)
  HRV: Croatia
  HUN: Hungary
  IDN: Indonesia
  IRL: Ireland
  ISL: Iceland
  ISR: Israel
  ITA: Italy
  JAM: Jamaica
  JOR: Jordan
  JPN: Japan
  KAZ: Kazakhstan
  KHM: Cambodia
  KOR: Korea
  KSV: Kosovo
  LTU: Lithuania
  LVA: Latvia
  MAC: Macao (China)
  MAR: Morocco
  MDA: Republic of Moldova
  MEX: Mexico
  MKD: North Macedonia
  MLT: Malta
  MNE: Montenegro
  MNG: Mongolia
  MYS: Malaysia
  NLD: Netherlands
  NOR: Norway
  NZL: New Zealand
  PAN: Panama
  PER: Peru
  PHL: Philippines
  P

In [6]:
# Cognitive dosyasındaki değişkenleri türüne göre kategorize edelim
# PISA'da process data değişkenleri belli ön ekler/son ekler kullanır

time_vars = []        # Süre değişkenleri (T kelimesi içerir genelde)
step_vars = []        # Adım sayıları
response_vars = []    # Cevap değişkenleri
score_vars = []       # Puanlar
other_vars = []

for name in meta_cog.column_names:
    label = meta_cog.column_names_to_labels.get(name, "") or ""
    label_lower = label.lower()
    
    if "time" in label_lower or name.endswith("T") and len(name) > 2:
        time_vars.append((name, label))
    elif "step" in label_lower or "action" in label_lower:
        step_vars.append((name, label))
    elif name.endswith("C") and "response" in label_lower:
        response_vars.append((name, label))
    elif name.endswith("S") and "score" in label_lower:
        score_vars.append((name, label))
    else:
        other_vars.append((name, label))

print(f"Süre ile ilgili değişkenler: {len(time_vars)}")
print(f"Adım/action değişkenleri: {len(step_vars)}")
print(f"Cevap değişkenleri: {len(response_vars)}")
print(f"Puan değişkenleri: {len(score_vars)}")
print(f"Diğer: {len(other_vars)}")

print("\n--- İlk 10 süre değişkeni örneği: ---")
for name, label in time_vars[:10]:
    print(f"  {name}: {label}")

Süre ile ilgili değişkenler: 1194
Adım/action değişkenleri: 533
Cevap değişkenleri: 278
Puan değişkenleri: 729
Diğer: 2289

--- İlk 10 süre değişkeni örneği: ---
  CNT: Country code 3-character
  Option_CT: Creative Thinking Option
  RCORE_TEST: Reading Adaptive Testlet Assignment: Core
  RS1_TEST: Reading Adaptive Testlet Assignment: Stage1
  RS2_TEST: Reading Adaptive Testlet Assignment: Stage2
  MCORE_TEST: Math Adaptive Testlet Assignment: Core
  MS1_TEST: Math Adaptive Testlet Assignment: Stage1
  MS2_TEST: Math Adaptive Testlet Assignment: Stage2
  CM033Q01TT: A View Room - Q01 (Total Timing)
  CM033Q01F: A View Room - Q01 (Time to First Action)


In [7]:
# PISA'da alan kodları: M=Math, R=Reading, S=Science, CT=Creative Thinking, FL=Financial Literacy
# Soru ID'leri genelde "CM001Q01T" (math), "CR001Q01T" (reading) gibi başlar

math_items = [n for n in meta_cog.column_names if n.startswith("CM") or n.startswith("DM")]
read_items = [n for n in meta_cog.column_names if n.startswith("CR") or n.startswith("DR")]
science_items = [n for n in meta_cog.column_names if n.startswith("CS") or n.startswith("DS")]
ct_items = [n for n in meta_cog.column_names if n.startswith("CC")]  # Creative Thinking
fl_items = [n for n in meta_cog.column_names if n.startswith("CF")]  # Financial Literacy

print(f"Matematik item sayısı: {len(math_items)}")
print(f"Okuma item sayısı: {len(read_items)}")
print(f"Fen item sayısı: {len(science_items)}")
print(f"Creative Thinking item sayısı: {len(ct_items)}")
print(f"Financial Literacy item sayısı: {len(fl_items)}")

print("\n--- Matematik değişken örnekleri ---")
for name in math_items[:15]:
    label = meta_cog.column_names_to_labels.get(name, "")
    print(f"  {name}: {label}")

Matematik item sayısı: 1654
Okuma item sayısı: 1592
Fen item sayısı: 873
Creative Thinking item sayısı: 0
Financial Literacy item sayısı: 0

--- Matematik değişken örnekleri ---
  CM033Q01S: A View Room - Q01 (Scored Response)
  DM033Q01R: A View Room - Q01 (Raw Response)
  CM033Q01TT: A View Room - Q01 (Total Timing)
  CM033Q01F: A View Room - Q01 (Time to First Action)
  CM033Q01A: A View Room - Q01 (Number of Actions)
  CM033Q01V: A View Room - Q01 (Number of Visits)
  CM033Q01VS: A View Room - Q01 (Number of Short Visits)
  CM474Q01S: Running Time - Q01 (Scored Response)
  DM474Q01R: Running Time - Q01 (Raw Response)
  CM474Q01TT: Running Time - Q01 (Total Timing)
  CM474Q01F: Running Time - Q01 (Time to First Action)
  CM474Q01A: Running Time - Q01 (Number of Actions)
  CM474Q01V: Running Time - Q01 (Number of Visits)
  CM474Q01VS: Running Time - Q01 (Number of Short Visits)
  DM155Q02C: Population Pyramids - Q02 (Coded Response)


In [8]:
# Hangi sütunları yükleyeceğiz?
# 1. Kimlik sütunları (her analiz için gerekli)
id_cols = ['CNT', 'CNTRYID', 'CNTSCHID', 'CNTSTUID', 'BOOKID',
           'LANGTEST_COG', 'ADMINMODE']

# 2. Matematik adaptive testing bilgileri (hangi aşamaya girdiği)
math_design_cols = [n for n in meta_cog.column_names 
                    if n.startswith('M') and 'TEST' in n or n.startswith('MS') and 'LEV' in n]

# 3. Tüm matematik item değişkenleri (S, TT, F, A, V, VS son ekli olanlar)
math_item_cols = [n for n in meta_cog.column_names 
                  if (n.startswith('CM') or n.startswith('DM'))]

# Hepsini birleştir, tekrar edenleri çıkar
cols_to_load = list(set(id_cols + math_design_cols + math_item_cols))

print(f"Toplam yüklenecek sütun sayısı: {len(cols_to_load)}")
print(f"  Kimlik sütunları: {len(id_cols)}")
print(f"  Math adaptive design: {len(math_design_cols)}")
print(f"  Math item değişkenleri: {len(math_item_cols)}")

Toplam yüklenecek sütun sayısı: 1666
  Kimlik sütunları: 7
  Math adaptive design: 5
  Math item değişkenleri: 1654


In [9]:
import time

print("Cognitive dosyası yükleniyor... (Birkaç dakika sürebilir)")
start = time.time()

df_cog, _ = pyreadstat.read_sav(
    str(cognitive_file),
    usecols=cols_to_load,
    disable_datetime_conversion=True  # hız için
)

elapsed = time.time() - start
print(f"\n✓ Yükleme tamamlandı: {elapsed/60:.1f} dakika")
print(f"Satır sayısı: {len(df_cog):,}")
print(f"Sütun sayısı: {df_cog.shape[1]}")
print(f"Bellek kullanımı: {df_cog.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

Cognitive dosyası yükleniyor... (Birkaç dakika sürebilir)

✓ Yükleme tamamlandı: 1.3 dakika
Satır sayısı: 613,744
Sütun sayısı: 1666
Bellek kullanımı: 7.62 GB


In [10]:
# İlk 5 satıra bakalım — sadece birkaç sütunla (çünkü 1660 sütunu sığmaz)
print("Ülke dağılımı (ilk 10):")
print(df_cog['CNT'].value_counts().head(10))

print(f"\nToplam öğrenci sayısı: {len(df_cog):,}")
print(f"Ülke sayısı: {df_cog['CNT'].nunique()}")

Ülke dağılımı (ilk 10):
CNT
ESP    30800
ARE    24600
CAN    23073
KAZ    19769
IDN    13439
AUS    13437
GBR    12972
ARG    12111
BRA    10798
ITA    10552
Name: count, dtype: int64

Toplam öğrenci sayısı: 613,744
Ülke sayısı: 80


In [11]:
# processed klasörünü oluştur (yoksa)
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

# Parquet dosyasına yaz (çok daha küçük ve hızlı okunur)
output_file = processed_dir / "pisa2022_math_cognitive.parquet"

print("Parquet formatına dönüştürülüyor...")
start = time.time()
df_cog.to_parquet(output_file, compression='snappy')
elapsed = time.time() - start

size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Kaydedildi: {output_file.name}")
print(f"  Boyut: {size_mb:.1f} MB (orijinal SAV: 3560 MB)")
print(f"  Süre: {elapsed:.1f} saniye")

Parquet formatına dönüştürülüyor...
✓ Kaydedildi: pisa2022_math_cognitive.parquet
  Boyut: 329.4 MB (orijinal SAV: 3560 MB)
  Süre: 23.1 saniye


In [12]:
import time

# Parquet'ten yeniden okuyalım - ne kadar sürdüğünü görelim
print("Parquet'ten okuma testi...")
start = time.time()
df_test = pd.read_parquet(output_file)
elapsed = time.time() - start

print(f"✓ Okuma süresi: {elapsed:.1f} saniye")
print(f"  Satır: {len(df_test):,}")
print(f"  Sütun: {df_test.shape[1]}")
print(f"  RAM: {df_test.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

del df_test  # Test DataFrame'ini silelim, RAM'i koruyalım

Parquet'ten okuma testi...
✓ Okuma süresi: 9.8 saniye
  Satır: 613,744
  Sütun: 1666
  RAM: 7.62 GB


In [13]:
# Öğrenci dosyasından metadata'yı yeniden okuyalım (notebook baştan çalıştıysa meta_stu hazır)
# Eğer meta_stu yoksa:
if 'meta_stu' not in dir():
    _, meta_stu = pyreadstat.read_sav(str(student_file), metadataonly=True)

# Kritik arka plan değişkenleri
key_student_vars = [
    # Kimlik
    'CNT', 'CNTRYID', 'CNTSCHID', 'CNTSTUID',
    # Demografik
    'ST001D01T',   # Grade
    'ST003D02T',   # Birth month
    'ST003D03T',   # Birth year  
    'ST004D01T',   # Gender
    'AGE',         # Age
    # Sosyoekonomik ve aile
    'ESCS',        # Index of economic, social and cultural status
    'HISEI',       # Highest parental occupational status
    'PAREDINT',    # Highest education of parents (years)
    'HOMEPOS',     # Home possessions index
    # Dil
    'LANGN',       # Language at home
    'IMMIG',       # Immigration status
    # Okul
    'SCHLTYPE',    # School type
    # Ağırlıklar (istatistik için kritik!)
    'W_FSTUWT',    # Final student weight
]

# Plausible values — matematik yetenek tahminleri (her öğrenci için 10 adet)
math_pvs = [f'PV{i}MATH' for i in range(1, 11)]  # PV1MATH, PV2MATH, ..., PV10MATH
read_pvs = [f'PV{i}READ' for i in range(1, 11)]
scie_pvs = [f'PV{i}SCIE' for i in range(1, 11)]

# Replicate weights (istatistik hesaplamaları için — 80 adet)
rep_weights = [f'W_FSTURWT{i}' for i in range(1, 81)]

# Hepsini birleştir ve sadece var olanları seç
student_cols = key_student_vars + math_pvs + read_pvs + scie_pvs + rep_weights
student_cols = [c for c in student_cols if c in meta_stu.column_names]

print(f"Toplam seçilen değişken: {len(student_cols)}")
print(f"  Temel değişkenler: {sum(1 for c in key_student_vars if c in meta_stu.column_names)}")
print(f"  Plausible values (3 alan × 10): {sum(1 for c in math_pvs + read_pvs + scie_pvs if c in meta_stu.column_names)}")
print(f"  Replicate weights: {sum(1 for c in rep_weights if c in meta_stu.column_names)}")

Toplam seçilen değişken: 126
  Temel değişkenler: 16
  Plausible values (3 alan × 10): 30
  Replicate weights: 80


In [14]:
print("Student dosyası yükleniyor... (5-10 dakika sürebilir)")
start = time.time()

df_stu, _ = pyreadstat.read_sav(
    str(student_file),
    usecols=student_cols,
    disable_datetime_conversion=True
)

elapsed = time.time() - start
print(f"✓ Yükleme tamamlandı: {elapsed/60:.1f} dakika")
print(f"Satır sayısı: {len(df_stu):,}")
print(f"Sütun sayısı: {df_stu.shape[1]}")
print(f"Bellek: {df_stu.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

# Parquet'e kaydet
stu_output = processed_dir / "pisa2022_student_background.parquet"
df_stu.to_parquet(stu_output, compression='snappy')

size_mb = stu_output.stat().st_size / (1024 * 1024)
print(f"\n✓ Parquet kaydedildi: {stu_output.name}")
print(f"  Boyut: {size_mb:.1f} MB")

Student dosyası yükleniyor... (5-10 dakika sürebilir)
✓ Yükleme tamamlandı: 0.3 dakika
Satır sayısı: 613,744
Sütun sayısı: 126
Bellek: 0.58 GB

✓ Parquet kaydedildi: pisa2022_student_background.parquet
  Boyut: 228.4 MB
